## `kube-apiserver` — the front door

The API server is the **only** thing that writes to `etcd`, and the **only** thing every other component talks to. Kill it and workloads keep running where they are, but the cluster makes no new decisions.

Every request — `kubectl get`, the scheduler's watch, the kubelet's status patch, your CI apply — passes five stages:

```
AUTHN → AUTHZ → ADMISSION → VALIDATION → etcd
who?    can?    mutate+     schema
        (RBAC)  validate    check
```

1. **AUTHN** — *who?* Tries each authenticator (client certs, ServiceAccount tokens, OIDC, webhook); first match wins. No match → **401**.
2. **AUTHZ** — *can they?* Usually **RBAC** (notebook 10): verb × object against bound roles. Denied → **403**.
3. **Admission** — allowed, but should it be *modified* or *policy-rejected*? **Mutating** controllers change the request (`ServiceAccount` token mount, `LimitRanger` defaults); **validating** ones reject (`ResourceQuota`, `PodSecurity`, Gatekeeper/Kyverno webhooks).
4. **Validation** — final schema check; bad → **422**.
5. **Persistence** — the (maybe mutated) object is written to `etcd`; watchers fire.

This is your debugging map: 401 = bad credential, 403 = missing RBAC, webhook error = an operator rejected you, 422 = wrong field/version. `kubectl --v=8` shows the full HTTP.

Flags the CKA expects you to recognise (all in `/etc/kubernetes/manifests/kube-apiserver.yaml`): `--etcd-servers`, `--service-cluster-ip-range`, `--authorization-mode=Node,RBAC`, `--enable-admission-plugins`, `--client-ca-file`/`--tls-cert-file`. On our map this is the **kube-apiserver** — the center of the control plane, the door in that `AUTHN→AUTHZ→admission→validation` sequence guards.